# QDArchive Data Acquisition Pipeline ✓ Full Run

This notebook orchestrates the complete data acquisition pipeline:
1. **Initialization** ✓ database, clients, collector
2. **Collection** ✓ query Zenodo and Dryad with QDA and qualitative keywords
3. **Download** ✓ stream files to local storage
4. **Export** ✓ persist metadata to CSV
5. **Report** ✓ summary stats and QA

## 1. Setup & Imports

In [1]:
import sys
from pathlib import Path
import pandas as pd
from datetime import datetime

# Add project root to path
project_root = Path.cwd()
sys.path.insert(0, str(project_root))

# Pipeline components
from params.config import (
    ZENODO_API,
    DRYAD_API,
    FIGSHARE_API,
    QUERIES_QDA,
    QUERIES_QUAL_EN,
    QUERIES_MULTILINGUAL_ALL,
    ALL_QUERIES_EN,
    ALL_QUERIES,
)
from clients.zenodo_client import ZenodoClient
from clients.dryad_client import DryadClient
from clients.cessda_client import CESSDAClient
from clients.figshare_client import FigshareClient
from pipeline.collector import PipelineCollector
from pipeline.database import QDArchDatabase
from pipeline.downloader import DatasetDownloader

print("✓ All imports successful")

✓ All imports successful


## 2. Database Initialization

Create or open the SQLite database. The schema includes:
- **datasets** ✓ metadata for each record
- **files** ✓ file listings with download URLs

In [2]:
db_path = "database/23100834-seeding.db"
db = QDArchDatabase(db_path=db_path)

print(f"Database initialized at {db_path}")

# Show current state
try:
    df_datasets = db.query("SELECT COUNT(*) as count FROM datasets")
    df_files = db.query("SELECT COUNT(*) as count FROM files")
    print(f"  Current: {df_datasets.iloc[0]['count']} dataset records, {df_files.iloc[0]['count']} files")
except:
    print("  (Fresh database)")

Database initialized at database/23100834-seeding.db
  (Fresh database)


## 3. Client Initialization

Set up API clients for Zenodo and Dryad.

**Optional:** If you have a Zenodo personal token (from https://zenodo.org/account/settings/applications/), set it below to increase the page size from 25 to 100+.

In [3]:
# Initialize clients
zenodo_client = ZenodoClient(
    base_url=ZENODO_API,
    access_token="0vhcRrWqQtZVWeT7m8mNI0rnnyIoB4wXSDFQx1iMzgxolGgw6CZcrL2fiXqX",
    timeout=60,
)

dryad_client = DryadClient(
    base_url=DRYAD_API,
    timeout=60,
    file_fetch_delay=0.5,
)

cessda_client = CESSDAClient(
    timeout=60,
    metadata_language="en",
)

figshare_client = FigshareClient(
    base_url=FIGSHARE_API,
    timeout=60,
    file_fetch_delay=0.3,
)

# Toggle repositories on/off here
include_cessda = True  # Set to True to include CESSDA
include_figshare = True  # Set to True to include Figshare

clients = [zenodo_client, dryad_client]
if include_cessda:
    clients.append(cessda_client)
if include_figshare:
    clients.append(figshare_client)

print(f"  ✓ Initialized {len(clients)} clients:")
print(f"    - ZenodoClient (page size: {zenodo_client.max_size})")
print(f"    - DryadClient")
if include_cessda:
    print(f"    - CESSDAClient (download_method: {cessda_client.download_method})")
if include_figshare:
    print(f"    - FigshareClient (download_method: {figshare_client.download_method})")

  ✓ Initialized 4 clients:
    - ZenodoClient (page size: 100)
    - DryadClient
    - CESSDAClient (download_method: SCRAPING)
    - FigshareClient (download_method: API-CALL)


## About CESSDA

**CESSDA** (Consortium of European Social Science Data Archives) is a metadata catalog of social science research datasets, primarily surveys and statistical studies. 

### Key Characteristics:
- **~38,000 datasets** available in the catalog
- **Metadata-only** — no direct file downloads (links to publisher archives)
- **Access levels** — Most datasets are restricted; only "Open" marked datasets are included
- **License parsing** — Extracts CC licenses from access descriptions; defaults to CC-BY
- **Download method** — `SCRAPING` (uses undocumented internal REST API)
- **Relevance** — Primarily social science (surveys, statistics); limited QDA software content

### When to Use CESSDA:
- ✓ Broader coverage of European qualitative research (interviews, focus groups in social science)
- ✓ Institutional datasets from CESSDA member archives
- ✗ Limited for QDA software exports (.qdpx, .nvpx) compared to Zenodo/Dryad

To disable CESSDA collection, set `include_cessda = False` above.

## About Figshare

**Figshare** is one of the world's largest open research data repositories, hosting millions of diverse research outputs including datasets, code, preprints, and media.

### Key Characteristics:
- **40M+ articles** available in the open repository
- **Free-text search** on title, description, and tags (title keywords preferred for relevance)
- **Open licenses** — Predominantly CC-BY-4.0, CC-BY-SA, CC0, GPL licenses
- **Download method** — `API-CALL` (public REST API v2)
- **File support** — Full file download URLs available per article
- **Metadata** — Rich structured metadata (DOI, authors, publication dates, tags, descriptions)
- **Relevance** — Mixed content; includes interview transcripts, qualitative datasets, and research materials

### When to Use Figshare:
- ✓ Large-scale coverage of open research data across all disciplines
- ✓ High-quality structured metadata and file support
- ✓ Good for interviews, focus group transcripts, and qualitative studies
- ✗ Lower concentration of QDA software exports compared to Zenodo
- ✗ Requires fetching full article details (slower than simple search APIs)

To disable Figshare collection, set `include_figshare = False` above.

## 4. Collection Configuration

Choose which queries to run. Options:
- `QUERIES_QDA` — just format/software names (narrow, high precision)
- `QUERIES_QUAL_EN` — qualitative methodology terms (broader)
- `ALL_QUERIES_EN` — both (recommended starting point)
- `ALL_QUERIES` — includes multilingual queries (larger scope)

In [4]:
# Choose your query set
queries_to_run = ["'interview'"]  # ~1 query (for quick testing)
# queries_to_run = QUERIES_QDA  # ~14 queries
# queries_to_run = QUERIES_QUAL_EN  # ~20 queries
# queries_to_run = ALL_QUERIES_EN  # ~34 queries (recommended)
# queries_to_run = ALL_QUERIES  # ~68 queries (all languages)

print(f"✓ Will run {len(queries_to_run)} queries")
print(f"First 5 queries:")
for i, q in enumerate(queries_to_run[:5]):
    print(f"  {i+1}. {q}")
print(f"  ...")

✓ Will run 1 queries
First 5 queries:
  1. 'interview'
  ...


## 5. Initialize Collector & Run Collection

Create the collector and begin querying. This will:
- Query each client for each search term
- Filter by open license + relevance score
- Persist results to the database
- Deduplicate across sources

In [5]:
collector = PipelineCollector(
    clients=clients,
    db=db,
    request_delay=1.0,  # seconds between API calls
)

print("Starting collection...")
start_time = datetime.now()

# Collect with pagination
records = collector.collect_multi_query(
    queries=queries_to_run,
    max_pages=3,      # pages per query
    page_size=25,     # results per page
    min_relevance=1,  # minimum score to keep a record
)

elapsed = datetime.now() - start_time
print(f"✓ Collection completed in {elapsed}")
print(f"  Collected {len(records)} unique relevant records")

Starting collection...
  query: "'interview'"
  [429] retrying in 1s (attempt 1/3)
  [429] retrying in 2s (attempt 2/3)
  [429] retrying in 4s (attempt 3/3)
  [dryad] could not fetch files from https://datadryad.org/api/v2/versions/100676/files: 
  [429] retrying in 1s (attempt 1/3)
Collected 158 unique relevant records.
✓ Collection completed in 0:04:07.222159
  Collected 158 unique relevant records


## 6. Inspect Collection Results

In [6]:
# Get statistics
stats = db.get_stats()

print(f"Database Statistics:")
print(f"  Total projects:  {stats['total_projects']}")
print(f"  Total files:     {stats['total_files']}")
print(f"  Total keywords:  {stats['total_keywords']}")
print(f"  Total people:    {stats['total_people']}")
print(f"  Total licenses:  {stats['total_licenses']}")

print(f"Files by download status:")
for status, count in stats['files_by_status'].items():
    pct = 100 * count / stats['total_files'] if stats['total_files'] > 0 else 0
    print(f"  {status:15s} {count:4d} ({pct:5.1f}%)")

Database Statistics:
  Total projects:  158
  Total files:     491
  Total keywords:  753
  Total people:    385
  Total licenses:  158
Files by download status:
  pending          491 (100.0%)


## 7. View Projects

In [7]:
# Show recent projects
projects = db.get_projects()
print(f"Total projects: {len(projects)}")

if len(projects) > 0:
    sample_cols = ['id', 'title', 'repository_url', 'doi', 'upload_date']
    print(projects[sample_cols].head(10).to_string())
else:
    print("(No projects yet)")

Total projects: 158
    id                                                                                                                                                                                                                   title        repository_url                                  doi upload_date
0  158                                                                                                                                           Table 1_Best practices along the kidney transplantation clinical journey.docx  https://figshare.com      10.3389/frtra.2026.1779662.s001  2026-04-13
1  157  Supplementary information files for "Perspectives towards, and experiences of clean sport in international Cerebral Palsy Football: A cross-cultural qualitative exploration of players and athlete support personnel"  https://figshare.com        10.17028/rd.lboro.31995969.v1  2026-04-13
2  156                                                                                            

## 8. Download Phase

Download files for all projects with pending status.

In [ ]:
downloader = DatasetDownloader(
    db=db,
    output_dir="downloads",
    request_delay=1.0,
    timeout=120,
    max_file_size_mb=500.0,
)

print("Starting downloads...")
start_time = datetime.now()

download_stats = downloader.download_all(resume=True)

elapsed = datetime.now() - start_time
print(f"  Download completed in {elapsed}")
print(f"  Success: {download_stats['success']}")
print(f"  Failed: {download_stats['failed']}")
print(f"  Skipped: {download_stats['skipped']}")

## 8b. Download Only ZIP Files

Alternative: Download only zip archive files (full datasets).
This is useful for downloading complete project bundles.

In [ ]:
# Download only ZIP files (full dataset archives)
print("Downloading ZIP files only...")
start_time = datetime.now()

# Filter to only files with .zip extension
zip_download_stats = downloader.download_all(
    status_filter="pending",
    resume=True,
    extensions={"zip"}  # Only download ZIP files
)

elapsed = datetime.now() - start_time
print(f"  ZIP download completed in {elapsed}")
print(f"  Success: {zip_download_stats['success']}")
print(f"  Failed: {zip_download_stats['failed']}")
print(f"  Skipped: {zip_download_stats['skipped']}")

## 9. Export Metadata to CSV

In [ ]:
db.export_projects_csv(path="exports/projects.csv")
db.export_files_csv(path="exports/files.csv")

print("✓ Exported:")
print("  exports/projects.csv")
print("  exports/files.csv")

## 10. Final Report

In [ ]:
stats = db.get_stats()

print("" + "="*70)
print("FINAL REPORT")
print("="*70)
print(f"Total projects collected: {stats['total_projects']}")
print(f"Total files recorded:     {stats['total_files']}")
print(f"Metadata files exported:")
print(f"  exports/projects.csv")
print(f"  exports/files.csv")
print(f"Database location: {db.db_path}")
print("" + "="*70)